High-Dimensional Dynamic Factor Models (DFM) & Covariance Shrinkage for Cross-Sectional Alpha
* Marchenko-Pastur to separate noise from macro statistical factors
* Implemented non-linear covariance shrinkage (Ledoit-Wolf / OAS) to eliminate ill-conditioning in high-dimensional return matrices.
* Extracted residual idiosyncratic return dynamics via PCA factor suppression to strip market noise variance.

### Import libraries

In [28]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.covariance import LedoitWolf, OAS
import warnings
warnings.filterwarnings('ignore')

### 1. Data Ingestion & Universe Alignment

In [29]:
tickers = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'JNJ',
    'WMT', 'PG', 'UNH', 'HD', 'MA', 'BAC', 'XOM', 'PFE', 'KO', 'PEP',
    'COST', 'CSCO', 'ABT', 'ORCL', 'CVX', 'ACN', 'MCD', 'DIS', 'LIN', 'BMY',
    'INTC', 'TXN', 'AMT', 'NOW', 'QCOM', 'UNP', 'PBD', 'SPGI', 'HON', 'GE'
]

# Pull data
raw_data = yf.download(tickers, period="2y", interval="1d")

# Robust extraction of Close or Adj Close prices across yfinance formats
if 'Adj Close' in raw_data.columns.levels[0] if isinstance(raw_data.columns, pd.MultiIndex) else 'Adj Close' in raw_data.columns:
    data = raw_data['Adj Close']
else:
    data = raw_data['Close']

# Drop columns that failed to download or are entirely empty
data = data.dropna(how='all', axis=1)

# Compute daily log returns
returns = np.log(data / data.shift(1)).dropna()

N = returns.shape[1]  # Number of assets
T = returns.shape[0]  # Number of observations
q = N / T             # Aspect ratio
print("Fetching S&P 500 Constituent Data...")
print(f"Data Loaded: Assets (N) = {N}, Time Steps (T) = {T}, Aspect Ratio (q) = {q:.4f}")

[*********************100%***********************]  40 of 40 completed

Fetching S&P 500 Constituent Data...
Data Loaded: Assets (N) = 40, Time Steps (T) = 501, Aspect Ratio (q) = 0.0798


### 2. Marchenko-Pastur Analytical Fitting
Computing Empirical Eigenvalues & Marchenko-Pastur Bounds.
$$\lambda_\pm = \sigma^2 (1\pm\sqrt{q})^2$$

In [30]:
# Sample Correlation Matrix
C_sample = returns.corr().values

# Spectral Decomposition
eigenvalues, eigenvectors = np.linalg.eigh(C_sample)
# Sort eigenvalues in descending order
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Marchenko-Pastur Theoretical Bounds (assuming variance sigma^2 = 1 for correlation matrix)
sigma_sq = 1.0
lambda_minus = sigma_sq * (1.0 - np.sqrt(q))**2
lambda_plus = sigma_sq * (1.0 + np.sqrt(q))**2

print(f"Marchenko-Pastur Theoretical Upper Noise Bound (λ+): {lambda_plus:.4f}")
print(f"Marchenko-Pastur Theoretical Lower Noise Bound (λ-): {lambda_minus:.4f}")

Marchenko-Pastur Theoretical Upper Noise Bound (λ+): 1.6450
Marchenko-Pastur Theoretical Lower Noise Bound (λ-): 0.5147


#### Identify Market/Signal Factors vs Noise

In [31]:
signals = eigenvalues[eigenvalues > lambda_plus]
noise = eigenvalues[eigenvalues <= lambda_plus]
print(f"Detected Market/Systematic Factors: {len(signals)} | Pure Noise Components: {len(noise)}")

Detected Market/Systematic Factors: 4 | Pure Noise Components: 36


### 3. RMT Noise Filtering (Eigenvalue Clipping)
Cleaning Correlation Matrix via RMT Clipping

In [32]:
eigenvalues_cleaned = eigenvalues.copy()
# Replace noise eigenvalues with their mean to preserve matrix trace (sum of eigenvalues)
mean_noise_eigenvalue = np.mean(noise)
eigenvalues_cleaned[eigenvalues <= lambda_plus] = mean_noise_eigenvalue

# Reconstruct Cleaned Correlation Matrix C_clean = V * Λ_clean * V^T
C_clean = eigenvectors @ np.diag(eigenvalues_cleaned) @ eigenvectors.T
# Renormalize diagonal to 1.0 (valid correlation matrix structure)
inv_diag = 1.0 / np.sqrt(np.diag(C_clean))
C_clean = np.outer(inv_diag, inv_diag) * C_clean

### 4. Covariance Shrinkage (Ledoit-Wolf & OAS)
Computing Ledoit-Wolf and OAS Shrinkage Matrices

Condition number is used to measure how stable a function is under small deviation of the input. For matrix $A$, it is
$$\kappa(A) = \lVert A\rVert\cdot \lVert A^{-1}\rVert.$$

Here the condition number after Ledoit-Wolf shrinkage is larger than the raw sample matrix.

In [33]:
lw = LedoitWolf().fit(returns)
oas = OAS().fit(returns)

cond_sample = np.linalg.cond(C_sample)
cond_clean = np.linalg.cond(C_clean)
cond_lw = np.linalg.cond(lw.covariance_)

print(f"Condition Number (Raw Sample Matrix): {cond_sample:.2f}")
print(f"Condition Number (RMT Cleaned Matrix): {cond_clean:.2f}")
print(f"Condition Number (Ledoit-Wolf Shrinkage): {cond_lw:.2f}")

Condition Number (Raw Sample Matrix): 84.52
Condition Number (RMT Cleaned Matrix): 17.23
Condition Number (Ledoit-Wolf Shrinkage): 87.16


### 5. Extract Idiosyncratic Residual Signals
Extracting Residual Signals via Principal Component Analysis.

In [34]:
# Use top K market factors identified by RMT
K = len(signals)
V_k = eigenvectors[:, :K]  # Top K eigenvector components

# Project standardized returns onto top K systematic factors
standardized_returns = (returns - returns.mean()) / returns.std()
systematic_returns = (standardized_returns.values @ V_k) @ V_k.T
idiosyncratic_residuals = standardized_returns.values - systematic_returns

# Convert to DataFrame
df_residuals = pd.DataFrame(idiosyncratic_residuals, index=returns.index, columns=returns.columns)

print("\n=== HIGH-DIMENSIONAL FACTOR MODELING RESULTS ===")
print(f"Market Noise Variance Stripped: {(1.0 - np.var(df_residuals.values) / np.var(standardized_returns.values)) * 100:.2f}%")
print(f"Residual Idiosyncratic Return Matrix Shape: {df_residuals.shape}")


=== HIGH-DIMENSIONAL FACTOR MODELING RESULTS ===
Market Noise Variance Stripped: 46.55%
Residual Idiosyncratic Return Matrix Shape: (501, 40)
